In [ ]:
import sys
sys.path.append('../')

import torch
import random
import torch.backends.cudnn as cudnn
import numpy as np
from utils.dataset_cylinder import GraphDataset_paired, GraphDataset_unpaired
from train.train_MGN_cylinder import train_MGN_comp, train_MGN_sup
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.cuda.manual_seed_all(0)
np.random.seed(0)
cudnn.benchmark = False
cudnn.deterministic = True
random.seed(0)

In [ ]:
device="cuda:0"

In [ ]:
data_dir="../data/data_cylinder/"
result_dir="../results/MGN_cylinder/"

In [ ]:
idx_list_test1=np.random.choice(list(range(800,1000)), 100, replace=False)
idx_list_train1_200=list(range(600,800))
idx_list_train1_40=np.random.choice(idx_list_train1_200,40, replace=False)
idx_list_train2_160=[i for i in idx_list_train1_200 if i not in idx_list_train1_40]

In [ ]:
test1=GraphDataset_paired(idx_list_test1, data_dir, device)

In [ ]:
train1=GraphDataset_paired(idx_list_train1_200, data_dir, device)
train_MGN_sup(device, train1, test1, result_dir, ib_n=False, ib_e=False, num_exp=1)
#sup: fully supervised baseline
#ib: inductive bias
#ib_n: node-level centering 
#ib_e: message-level centering
#num_exp: number of experiments with the same seed

In [ ]:
train1=GraphDataset_paired(idx_list_train1_40, data_dir, device)
train2=GraphDataset_unpaired(idx_list_train2_160, data_dir, device)

train_MGN_comp(device, train1, train2, test1, result_dir, ib_n=True, ib_e=True, num_exp=1)
#comp: complementary learning

In [ ]:
from utils.analysis import RMSE

print(RMSE(result_dir=result_dir, model="MGN", learning="sup", N_paired=200, N_total=200, ib="FF", exp_list=[0]))
print(RMSE(result_dir=result_dir, model="MGN", learning="comp", N_paired=40, N_total=200, ib="TT", exp_list=[0]))

#print (mean, std)